# Part 3: AI Agent for Credit Risk Report Generation

In this notebook, we'll create a multi-agent system using CrewAI to automatically generate comprehensive credit risk reports. Our agents will work together to analyze credit applications, assess risk, and produce professional reports.

## Learning Objectives
- Design and implement a multi-agent credit analysis system
- Create specialized agents for different aspects of credit risk assessment
- Generate automated, professional credit risk reports
- Understand agent collaboration and workflow orchestration
- Evaluate the effectiveness of agent-based approaches

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from datetime import datetime
import os
from typing import Dict, List, Any
import warnings
warnings.filterwarnings('ignore')

# CrewAI imports
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from langchain.tools import tool
from langchain_openai import ChatOpenAI

# For environment variables
from dotenv import load_dotenv
load_dotenv()

print("Libraries imported successfully!")
print(f"Current working directory: {os.getcwd()}")

## Load Data and Models

Let's load the credit application data and the trained model from previous notebooks:

In [ ]:
# Load credit applications dataset from JSON
try:
    with open('credit_applications_dataset.json', 'r') as f:
        dataset = json.load(f)
    
    df = pd.DataFrame(dataset['data'])
    print("✅ Dataset loaded successfully from JSON!")
    print(f"Dataset shape: {df.shape}")
    print(f"Default rate: {df['default_outcome'].mean():.2%}")
    
except FileNotFoundError:
    print("⚠️ JSON dataset not found. Please run notebook 1 first.")
    # Fallback to CSV if available
    try:
        df = pd.read_csv('credit_applications_dataset.csv')
        print("✅ Fallback: CSV dataset loaded")
    except FileNotFoundError:
        print("❌ No dataset found. Please run notebook 1 first.")
        df = None

# Load the trained model
try:
    model_artifacts = joblib.load('best_credit_risk_model.pkl')
    print("✅ Model loaded successfully!")
    print(f"Model type: {model_artifacts['model_name']}")
    print(f"Feature type: {model_artifacts['feature_type']}")
    print(f"AUC Score: {model_artifacts['performance']['auc_score']:.4f}")
except FileNotFoundError:
    print("⚠️ Model not found. Please run notebook 2 first.")
    model_artifacts = None

if df is not None:
    print(f"\nSample application IDs: {df['applicant_id'].head().tolist()}")
    print(f"Available columns: {list(df.columns)}")
    
    # Show sample data
    print("\nSample application:")
    sample_app = df.iloc[0]
    print(f"ID: {sample_app['applicant_id']}")
    print(f"Age: {sample_app['age']}, Income: ${sample_app['income']:,}")
    print(f"Loan: ${sample_app['loan_amount']:,} for {sample_app['purpose']}")
    print(f"Credit: {sample_app['credit_history']}")
    print(f"Description: {sample_app['text_description'][:100]}...")

## Create Custom Tools for Credit Analysis

Let's create custom tools that our agents can use to analyze credit applications:

In [ ]:
class CreditDataCollectionTool(BaseTool):
    """Tool for comprehensive credit data collection and validation"""
    
    name: str = "credit_data_collector"
    description: str = "Collects and validates comprehensive credit application data including cross-referencing multiple sources"
    
    def _run(self, applicant_id: str) -> str:
        """Collect comprehensive credit application data"""
        try:
            # Find the applicant in the dataset
            applicant_data = df[df['applicant_id'] == applicant_id]
            
            if applicant_data.empty:
                return f"No data found for applicant ID: {applicant_id}"
            
            # Extract relevant information
            data = applicant_data.iloc[0]
            
            # Simulate data validation checks
            validation_flags = []
            
            # Income validation
            if data['income'] < 15000:
                validation_flags.append("⚠️ Income below minimum threshold")
            
            # Employment validation
            if data['employment_length'] < 0.5:
                validation_flags.append("⚠️ Short employment history")
            
            # Debt validation
            if data['debt_to_income'] > 0.5:
                validation_flags.append("⚠️ High debt-to-income ratio")
            
            result = {
                'applicant_id': data['applicant_id'],
                'personal_info': {
                    'age': data['age'],
                    'location': data.get('location', 'Not specified'),
                    'education': data.get('education', 'Not specified')
                },
                'financial_info': {
                    'annual_income': data['income'],
                    'employment_length': data['employment_length'],
                    'debt_to_income_ratio': data['debt_to_income']
                },
                'loan_details': {
                    'requested_amount': data['loan_amount'],
                    'purpose': data['purpose'],
                    'loan_to_income_ratio': data['loan_amount'] / data['income']
                },
                'credit_profile': {
                    'credit_history': data['credit_history'],
                    'default_probability': data.get('default_probability', 'Not calculated'),
                    'predicted_outcome': data.get('predicted_token', 'Not available')
                },
                'application_narrative': data['text_description'],
                'data_validation': {
                    'flags': validation_flags,
                    'completeness_score': 0.95,  # Simulated
                    'data_quality': 'Good' if len(validation_flags) == 0 else 'Needs Review'
                }
            }
            
            return json.dumps(result, indent=2)
            
        except Exception as e:
            return f"Error collecting data: {str(e)}"

class RiskAssessmentTool(BaseTool):
    """Tool for comprehensive risk assessment and scoring"""
    
    name: str = "risk_assessment_calculator"
    description: str = "Performs comprehensive risk assessment including probability calculations, stress testing, and scenario analysis"
    
    def _run(self, applicant_id: str) -> str:
        """Calculate comprehensive risk assessment"""
        try:
            if df is None:
                return "Dataset not available"
            
            # Get applicant data
            applicant_data = df[df['applicant_id'] == applicant_id]
            
            if applicant_data.empty:
                return f"No data found for applicant ID: {applicant_id}"
            
            data = applicant_data.iloc[0]
            
            # Use the actual default probability from our model if available
            if 'default_probability' in data:
                base_probability = data['default_probability']
            else:
                # Fallback calculation
                base_probability = 0.05  # 5% base rate
                
                # Adjust based on credit history
                credit_multipliers = {'excellent': 0.3, 'good': 0.7, 'fair': 1.5, 'poor': 2.5}
                base_probability *= credit_multipliers.get(data['credit_history'], 1.0)
                
                # Adjust based on financial factors
                if data['debt_to_income'] > 0.4:
                    base_probability *= 1.5
                if data['loan_amount'] / data['income'] > 0.5:
                    base_probability *= 1.3
                
                base_probability = max(0.001, min(0.5, base_probability))
            
            # Risk categorization
            if base_probability < 0.05:
                risk_rating = "Low"
                decision_recommendation = "APPROVE"
            elif base_probability < 0.15:
                risk_rating = "Medium"
                decision_recommendation = "APPROVE with conditions"
            else:
                risk_rating = "High"
                decision_recommendation = "DECLINE or require additional collateral"
            
            # Stress testing scenarios
            stress_scenarios = {
                'economic_downturn': min(0.5, base_probability * 1.8),
                'interest_rate_increase': min(0.5, base_probability * 1.3),
                'income_reduction_10pct': min(0.5, base_probability * 1.4)
            }
            
            result = {
                'applicant_id': data['applicant_id'],
                'risk_assessment': {
                    'default_probability': round(base_probability, 4),
                    'risk_rating': risk_rating,
                    'decision_recommendation': decision_recommendation,
                    'confidence_level': 0.82  # Simulated
                },
                'key_risk_factors': [
                    f"Credit History: {data['credit_history']} (Weight: 35%)",
                    f"Debt-to-Income: {data['debt_to_income']:.2%} (Weight: 25%)",
                    f"Loan-to-Income: {data['loan_amount']/data['income']:.2%} (Weight: 20%)",
                    f"Employment Length: {data['employment_length']} years (Weight: 20%)"
                ],
                'stress_test_results': stress_scenarios,
                'model_details': {
                    'model_version': '2.1',
                    'last_updated': '2024-12-01',
                    'validation_auc': 0.85
                }
            }
            
            return json.dumps(result, indent=2)
            
        except Exception as e:
            return f"Error calculating risk assessment: {str(e)}"

class FinancialAnalysisTool(BaseTool):
    """Tool for detailed financial analysis and ratio calculations"""
    
    name: str = "financial_analysis_expert"
    description: str = "Performs comprehensive financial analysis including ratios, benchmarking, and scenario modeling"
    
    def _run(self, applicant_id: str) -> str:
        """Perform comprehensive financial analysis"""
        try:
            if df is None:
                return "Dataset not available"
            
            # Get applicant data
            applicant_data = df[df['applicant_id'] == applicant_id]
            
            if applicant_data.empty:
                return f"No data found for applicant ID: {applicant_id}"
            
            data = applicant_data.iloc[0]
            
            # Calculate comprehensive financial metrics
            monthly_income = data['income'] / 12
            loan_to_income = data['loan_amount'] / data['income']
            estimated_rate = 0.08  # 8% APR assumption
            estimated_monthly_payment = data['loan_amount'] * (estimated_rate / 12) / (1 - (1 + estimated_rate / 12) ** (-60))  # 5-year term
            payment_to_income = estimated_monthly_payment / monthly_income
            
            # Debt service coverage
            existing_debt_payment = monthly_income * data['debt_to_income']
            total_debt_service = existing_debt_payment + estimated_monthly_payment
            total_debt_service_ratio = total_debt_service / monthly_income
            
            # Industry benchmarks
            benchmarks = {
                'loan_to_income': {'excellent': 0.2, 'good': 0.3, 'fair': 0.4, 'poor': 0.5},
                'debt_to_income': {'excellent': 0.2, 'good': 0.28, 'fair': 0.36, 'poor': 0.43},
                'payment_to_income': {'excellent': 0.1, 'good': 0.15, 'fair': 0.2, 'poor': 0.25}
            }
            
            def assess_metric(value, benchmark):
                if value <= benchmark['excellent']:
                    return "Excellent"
                elif value <= benchmark['good']:
                    return "Good"
                elif value <= benchmark['fair']:
                    return "Fair"
                else:
                    return "Poor"
            
            # Calculate affordability score
            affordability_factors = {
                'income_stability': 0.9 if data['employment_length'] > 2 else 0.7,
                'debt_burden': 1.0 - data['debt_to_income'],
                'loan_size': 1.0 - min(0.5, loan_to_income),
                'payment_capacity': 1.0 - min(0.3, payment_to_income)
            }
            
            affordability_score = np.mean(list(affordability_factors.values()))
            
            result = {
                'applicant_id': data['applicant_id'],
                'financial_metrics': {
                    'monthly_income': round(monthly_income, 2),
                    'loan_to_income_ratio': round(loan_to_income, 3),
                    'debt_to_income_ratio': round(data['debt_to_income'], 3),
                    'estimated_monthly_payment': round(estimated_monthly_payment, 2),
                    'payment_to_income_ratio': round(payment_to_income, 3),
                    'total_debt_service_ratio': round(total_debt_service_ratio, 3),
                    'affordability_score': round(affordability_score, 2)
                },
                'benchmark_assessments': {
                    'loan_to_income': assess_metric(loan_to_income, benchmarks['loan_to_income']),
                    'debt_to_income': assess_metric(data['debt_to_income'], benchmarks['debt_to_income']),
                    'payment_to_income': assess_metric(payment_to_income, benchmarks['payment_to_income'])
                },
                'financial_strengths': [],
                'financial_concerns': [],
                'recommendations': [
                    f"Loan amount is {'appropriate' if loan_to_income <= 0.4 else 'high'} relative to income",
                    f"Debt levels are {'manageable' if data['debt_to_income'] <= 0.35 else 'concerning'}",
                    f"Employment history is {'stable' if data['employment_length'] >= 2 else 'limited'}"
                ]
            }
            
            # Add strengths and concerns
            if loan_to_income <= 0.3:
                result['financial_strengths'].append("Conservative loan-to-income ratio")
            if data['debt_to_income'] <= 0.28:
                result['financial_strengths'].append("Low existing debt burden")
            if data['employment_length'] >= 5:
                result['financial_strengths'].append("Strong employment stability")
            
            if loan_to_income > 0.4:
                result['financial_concerns'].append("High loan-to-income ratio")
            if data['debt_to_income'] > 0.35:
                result['financial_concerns'].append("Elevated debt-to-income ratio")
            if data['employment_length'] < 1:
                result['financial_concerns'].append("Limited employment history")
            
            return json.dumps(result, indent=2)
            
        except Exception as e:
            return f"Error performing financial analysis: {str(e)}"

class ComplianceCheckTool(BaseTool):
    """Tool for regulatory compliance verification"""
    
    name: str = "compliance_checker"
    description: str = "Ensures all decisions meet fair lending and banking regulations"
    
    def _run(self, applicant_id: str) -> str:
        """Perform compliance checks"""
        try:
            if df is None:
                return "Dataset not available"
            
            # Get applicant data
            applicant_data = df[df['applicant_id'] == applicant_id]
            
            if applicant_data.empty:
                return f"No data found for applicant ID: {applicant_id}"
            
            data = applicant_data.iloc[0]
            
            # Compliance checks
            compliance_checks = {
                'fair_lending': {
                    'protected_class_factors': 'Not considered in decision',
                    'disparate_impact': 'Within acceptable range',
                    'equal_treatment': 'Applied standard criteria'
                },
                'documentation': {
                    'income_verification': 'Required documentation present',
                    'identity_verification': 'ID documents validated',
                    'credit_authorization': 'Proper consent obtained'
                },
                'regulatory_limits': {
                    'loan_amount_limits': 'Within regulatory bounds',
                    'interest_rate_caps': 'Compliant with state regulations',
                    'fee_disclosures': 'Properly disclosed'
                }
            }
            
            # Risk-based compliance flags
            compliance_flags = []
            
            if data['debt_to_income'] > 0.43:
                compliance_flags.append("High DTI ratio - requires additional documentation")
            
            if data['loan_amount'] > 100000:
                compliance_flags.append("Large loan amount - requires enhanced due diligence")
            
            if data['employment_length'] < 0.5:
                compliance_flags.append("Limited employment history - verify income stability")
            
            # Overall compliance rating
            compliance_score = 0.95 if len(compliance_flags) == 0 else 0.85
            
            result = {
                'applicant_id': data['applicant_id'],
                'compliance_status': {
                    'overall_rating': 'Compliant' if compliance_score >= 0.9 else 'Needs Review',
                    'compliance_score': compliance_score,
                    'flags': compliance_flags
                },
                'regulatory_checks': compliance_checks,
                'audit_trail': {
                    'review_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                    'reviewer': 'AI Compliance Officer',
                    'regulations_applied': ['Fair Credit Reporting Act', 'Equal Credit Opportunity Act', 'Truth in Lending Act']
                },
                'recommendations': [
                    'Maintain current documentation standards',
                    'Continue monitoring for disparate impact',
                    'Ensure proper disclosures are provided'
                ]
            }
            
            return json.dumps(result, indent=2)
            
        except Exception as e:
            return f"Error performing compliance check: {str(e)}"

# Initialize tools
data_collection_tool = CreditDataCollectionTool()
risk_assessment_tool = RiskAssessmentTool()
financial_analysis_tool = FinancialAnalysisTool()
compliance_check_tool = ComplianceCheckTool()

print("✅ Custom tools created successfully!")
print("Available tools:")
print("  • Credit Data Collection Tool")
print("  • Risk Assessment Tool")
print("  • Financial Analysis Tool")
print("  • Compliance Check Tool")

## Define AI Agents

Now let's create specialized agents for different aspects of credit analysis:

In [ ]:
# Initialize the language model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1
)

# Agent 1: Senior Risk Analyst (Lead and Delegator)
senior_risk_analyst = Agent(
    role='Senior Risk Analyst',
    goal='Make final credit decisions on complex applications and coordinate the analysis team',
    backstory="""You are a Senior Risk Analyst with 15+ years of experience in credit risk management. 
    You are CRM certified and specialize in commercial lending. You have the authority to make final 
    credit decisions and coordinate a team of specialized analysts. You handle high-value applications 
    over $500K and can override decisions from other agents when needed. You only escalate to human 
    VP for policy violations.""",
    tools=[risk_assessment_tool],
    llm=llm,
    verbose=True,
    allow_delegation=True,  # This agent can delegate to others
    max_delegation=5
)

# Agent 2: Data Intelligence Specialist
data_intelligence_specialist = Agent(
    role='Data Intelligence Specialist',
    goal='Gather comprehensive, accurate applicant information and validate data quality',
    backstory="""You are a detail-oriented Data Intelligence Specialist with expertise in financial 
    data sources and validation. You specialize in pulling credit bureau reports, bank statements, 
    and employment verification. You cross-reference multiple data sources and flag inconsistencies 
    or missing information. Your work forms the foundation for all subsequent analysis.""",
    tools=[data_collection_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

# Agent 3: Financial Analysis Expert
financial_analysis_expert = Agent(
    role='Financial Analysis Expert',
    goal='Calculate precise risk metrics and financial ratios with mathematical rigor',
    backstory="""You are a Financial Analysis Expert with CFA certification and strong mathematical 
    background. You specialize in quantitative risk modeling, computing debt-to-income and loan-to-value 
    ratios, running stress tests and scenario analysis. You apply credit scoring models and generate 
    probability of default estimates with mathematical precision.""",
    tools=[financial_analysis_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

# Agent 4: Document Specialist
document_specialist = Agent(
    role='Document Specialist',
    goal='Create clear, compliant credit memos and decision documentation',
    backstory="""You are a professional Document Specialist and technical writer with extensive 
    banking compliance knowledge. You draft loan committee presentations, write credit decision 
    letters, and ensure regulatory compliance in all documentation. You maintain comprehensive 
    audit trails and create clear, consistent messaging for all stakeholders.""",
    tools=[],  # Primarily uses writing capabilities
    llm=llm,
    verbose=True,
    allow_delegation=False
)

# Agent 5: Customer Relations Manager
customer_relations_manager = Agent(
    role='Customer Relations Manager',
    goal='Maintain positive customer experience throughout the credit process',
    backstory="""You are a Customer Relations Manager with extensive customer service experience 
    and lending product knowledge. You handle applicant communication, send status updates, 
    explain decisions and next steps, and manage follow-up for approved loans. You ensure 
    positive customer experience regardless of the final decision.""",
    tools=[],  # Primarily uses communication capabilities
    llm=llm,
    verbose=True,
    allow_delegation=False
)

# Agent 6: Compliance Officer
compliance_officer = Agent(
    role='Compliance Officer',
    goal='Ensure all decisions meet fair lending and banking regulations',
    backstory="""You are a Compliance Officer and former bank examiner with deep regulatory 
    knowledge. You review all decisions for fair lending compliance, check that prohibited 
    factors aren't influencing decisions, maintain required documentation standards, and 
    generate regulatory reports. You have veto power over any decision for regulatory issues.""",
    tools=[compliance_check_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("✅ AI Agent Crew created successfully!")
print("\nAgent Structure:")
print("1. Senior Risk Analyst (Lead & Delegator)")
print("2. Data Intelligence Specialist")
print("3. Financial Analysis Expert")
print("4. Document Specialist")
print("5. Customer Relations Manager")
print("6. Compliance Officer")

## Define Tasks and Workflow

Let's create the tasks that our agents will perform:

In [ ]:
def create_credit_analysis_tasks(applicant_id: str):
    """Create tasks for credit analysis workflow following the delegation structure"""
    
    # Phase 1: Initial Assessment - Senior Risk Analyst delegates data collection
    data_collection_task = Task(
        description=f"""As delegated by the Senior Risk Analyst, collect comprehensive credit application 
        data for applicant {applicant_id}.
        
        Your responsibilities:
        1. Retrieve all available application data from multiple sources
        2. Validate data completeness and accuracy
        3. Cross-reference information for consistency
        4. Identify any missing or inconsistent information
        5. Flag potential data quality issues
        6. Organize data in structured format for analysis
        
        Focus areas:
        - Personal demographics and employment history
        - Financial details (income, assets, existing debts)
        - Credit history and payment patterns
        - Loan specifics and stated purpose
        - Supporting documentation status
        
        Report back to Senior Risk Analyst with comprehensive data summary.""",
        expected_output="Comprehensive, validated credit application data summary with quality flags",
        agent=data_intelligence_specialist
    )
    
    # Phase 1: Initial Assessment - Senior Risk Analyst delegates financial analysis
    financial_analysis_task = Task(
        description=f"""As delegated by the Senior Risk Analyst, conduct detailed financial analysis 
        for applicant {applicant_id}.
        
        Your responsibilities:
        1. Calculate all key financial ratios and metrics
        2. Perform stress testing and scenario analysis
        3. Apply credit scoring models and generate probability estimates
        4. Compare metrics against industry benchmarks
        5. Identify financial strengths and risk factors
        6. Provide quantitative risk assessment
        
        Key calculations required:
        - Debt-to-income and loan-to-value ratios
        - Monthly cash flow analysis
        - Payment capacity assessment
        - Default probability modeling
        - Stress test scenarios (economic downturn, rate changes)
        
        Report detailed financial analysis to Senior Risk Analyst.""",
        expected_output="Comprehensive financial analysis with risk metrics and stress test results",
        agent=financial_analysis_expert
    )
    
    # Phase 2: Decision Making - Senior Risk Analyst makes credit decision
    risk_decision_task = Task(
        description=f"""As Senior Risk Analyst, review all collected data and analysis to make 
        the final credit decision for applicant {applicant_id}.
        
        Your responsibilities:
        1. Review data collection and financial analysis results
        2. Assess overall credit risk using your experience
        3. Make final credit decision (Approve/Conditional/Decline)
        4. Provide detailed rationale for decision
        5. Identify any special conditions or requirements
        6. Determine if case needs human escalation
        
        Decision criteria:
        - Risk rating (Low/Medium/High)
        - Probability of default threshold
        - Policy compliance requirements
        - Regulatory considerations
        
        Prepare decision summary for documentation and communication.""",
        expected_output="Final credit decision with detailed rationale and conditions",
        agent=senior_risk_analyst,
        context=[data_collection_task, financial_analysis_task]
    )
    
    # Phase 2: Support Tasks - Senior Risk Analyst delegates documentation
    documentation_task = Task(
        description=f"""As delegated by the Senior Risk Analyst, create comprehensive documentation 
        for the credit decision on applicant {applicant_id}.
        
        Your responsibilities:
        1. Draft professional credit decision letter
        2. Create loan committee presentation if needed
        3. Ensure all documentation meets regulatory requirements
        4. Maintain complete audit trail
        5. Prepare internal risk memo
        6. Format reports for various stakeholders
        
        Documentation requirements:
        - Executive summary with key decision points
        - Detailed analysis supporting the decision
        - Risk assessment and mitigation measures
        - Regulatory compliance confirmation
        - Clear, professional language appropriate for clients
        
        Ensure all documentation is audit-ready and professionally formatted.""",
        expected_output="Professional credit decision documentation including decision letter and internal memo",
        agent=document_specialist,
        context=[risk_decision_task]
    )
    
    # Phase 2: Support Tasks - Senior Risk Analyst delegates customer communication
    customer_communication_task = Task(
        description=f"""As delegated by the Senior Risk Analyst, handle all customer communications 
        for applicant {applicant_id}.
        
        Your responsibilities:
        1. Prepare customer-friendly decision notification
        2. Explain decision rationale and next steps
        3. Handle any follow-up questions or concerns
        4. Manage additional documentation requests if needed
        5. Coordinate approval conditions and requirements
        6. Maintain positive customer relationship throughout process
        
        Communication principles:
        - Clear, transparent explanations
        - Professional and empathetic tone
        - Timely responses to inquiries
        - Proper escalation when needed
        
        Focus on customer satisfaction regardless of decision outcome.""",
        expected_output="Customer communication plan and decision notification with clear next steps",
        agent=customer_relations_manager,
        context=[risk_decision_task]
    )
    
    # Phase 2: Support Tasks - Senior Risk Analyst delegates compliance review
    compliance_review_task = Task(
        description=f"""As delegated by the Senior Risk Analyst, conduct comprehensive compliance 
        review for applicant {applicant_id} decision.
        
        Your responsibilities:
        1. Review decision for fair lending compliance
        2. Verify no prohibited factors influenced decision
        3. Check documentation meets regulatory standards
        4. Ensure proper audit trail maintenance
        5. Generate required regulatory reports
        6. Flag any compliance concerns or violations
        
        Compliance areas to review:
        - Fair Credit Reporting Act compliance
        - Equal Credit Opportunity Act adherence
        - Truth in Lending Act requirements
        - State and federal banking regulations
        
        You have veto power over decisions with regulatory issues.""",
        expected_output="Compliance review report with regulatory confirmation and any required actions",
        agent=compliance_officer,
        context=[risk_decision_task, documentation_task]
    )
    
    return [
        data_collection_task,
        financial_analysis_task,
        risk_decision_task,
        documentation_task,
        customer_communication_task,
        compliance_review_task
    ]

print("✅ Task creation function updated with proper delegation workflow!")
print("\nWorkflow Structure:")
print("Phase 1: Senior Risk Analyst delegates to:")
print("  - Data Intelligence Specialist")
print("  - Financial Analysis Expert")
print("Phase 2: Senior Risk Analyst makes decision, then delegates to:")
print("  - Document Specialist")
print("  - Customer Relations Manager")
print("  - Compliance Officer")

## Create and Run Credit Analysis Crew

Now let's create the crew and run the credit analysis for sample applications:

In [ ]:
def analyze_credit_application(applicant_id: str):
    """Run complete credit analysis using the 6-agent crew structure"""
    
    print(f"\n{'='*60}")
    print(f"CREDIT ANALYSIS FOR APPLICANT: {applicant_id}")
    print(f"{'='*60}")
    
    # Show applicant details first
    if df is not None:
        applicant_info = df[df['applicant_id'] == applicant_id].iloc[0]
        print(f"\nApplicant Overview:")
        print(f"  Age: {applicant_info['age']}")
        print(f"  Income: ${applicant_info['income']:,}")
        print(f"  Loan Amount: ${applicant_info['loan_amount']:,}")
        print(f"  Purpose: {applicant_info['purpose']}")
        print(f"  Credit History: {applicant_info['credit_history']}")
        print(f"  Employment: {applicant_info['employment_length']} years")
        print(f"  Debt-to-Income: {applicant_info['debt_to_income']:.2%}")
        print(f"  Description: {applicant_info['text_description'][:150]}...")
    
    # Create tasks following the delegation structure
    tasks = create_credit_analysis_tasks(applicant_id)
    
    # Create crew with all 6 agents
    crew = Crew(
        agents=[
            senior_risk_analyst,        # Lead agent with delegation authority
            data_intelligence_specialist, # Data collection
            financial_analysis_expert,   # Financial analysis
            document_specialist,         # Documentation
            customer_relations_manager,  # Customer communication
            compliance_officer          # Compliance review
        ],
        tasks=tasks,
        process=Process.sequential,
        verbose=True,
        manager_llm=llm  # Use LLM for crew coordination
    )
    
    # Execute the crew workflow
    try:
        print(f"\n🚀 Starting Credit Analysis Workflow...")
        print(f"   Phase 1: Data Collection & Financial Analysis")
        print(f"   Phase 2: Decision Making & Support Tasks")
        print(f"   Following delegation authority rules...")
        
        result = crew.kickoff()
        return result
        
    except Exception as e:
        print(f"❌ Error during crew execution: {str(e)}")
        return None

# Select sample applications for analysis
if df is not None:
    # Choose diverse sample applications
    sample_applicants = df['applicant_id'].sample(3, random_state=42).tolist()
    print(f"📋 Selected sample applicants: {sample_applicants}")
    
    # Analyze the first applicant
    first_applicant = sample_applicants[0]
    print(f"\n🔍 Analyzing applicant: {first_applicant}")
    
else:
    print("⚠️ No dataset available. Please run previous notebooks first.")
    first_applicant = None

In [ ]:
def display_crew_workflow():
    """Display the complete crew workflow and structure"""
    
    print(f"\n{'='*70}")
    print("AI AGENT CREW WORKFLOW STRUCTURE")
    print(f"{'='*70}")
    
    print(f"\n🎯 DELEGATION AUTHORITY RULES:")
    print(f"  • Senior Risk Analyst: CAN delegate to all specialists")
    print(f"  • Specialists: CANNOT delegate to peers")
    print(f"  • Support agents: Report back to Senior Risk Analyst")
    print(f"  • Compliance Officer: Has VETO power for regulatory issues")
    
    print(f"\n📋 PHASE 1: INITIAL ASSESSMENT")
    print(f"  Senior Risk Analyst delegates to:")
    print(f"  ├── Data Intelligence Specialist")
    print(f"  │   ├── Collect application data")
    print(f"  │   ├── Validate information")
    print(f"  │   └── Flag inconsistencies")
    print(f"  └── Financial Analysis Expert")
    print(f"      ├── Calculate risk metrics")
    print(f"      ├── Perform stress tests")
    print(f"      └── Generate probability estimates")
    
    print(f"\n🎯 PHASE 2: DECISION MAKING")
    print(f"  Senior Risk Analyst:")
    print(f"  ├── Reviews all analysis")
    print(f"  ├── Makes final credit decision")
    print(f"  └── Delegates support tasks to:")
    print(f"      ├── Document Specialist")
    print(f"      │   ├── Create decision letters")
    print(f"      │   ├── Prepare presentations")
    print(f"      │   └── Maintain audit trails")
    print(f"      ├── Customer Relations Manager")
    print(f"      │   ├── Notify applicants")
    print(f"      │   ├── Explain decisions")
    print(f"      │   └── Handle follow-ups")
    print(f"      └── Compliance Officer")
    print(f"          ├── Review for fair lending")
    print(f"          ├── Check documentation")
    print(f"          └── Generate regulatory reports")
    
    print(f"\n🔄 COMMUNICATION FLOW:")
    print(f"  Data Intelligence → Senior Risk Analyst")
    print(f"  Financial Analysis → Senior Risk Analyst")
    print(f"  Senior Risk Analyst → Document Specialist")
    print(f"  Senior Risk Analyst → Customer Relations")
    print(f"  Senior Risk Analyst → Compliance Officer")
    print(f"  Compliance Officer → [VETO if needed]")
    
    print(f"\n⚡ ADVANTAGES OF THIS STRUCTURE:")
    print(f"  • Clear chain of command")
    print(f"  • Specialized expertise")
    print(f"  • Parallel processing where possible")
    print(f"  • Built-in quality control")
    print(f"  • Regulatory compliance")
    print(f"  • Scalable architecture")

# Display the workflow
display_crew_workflow()

In [ ]:
# Run the credit analysis
analysis_result = analyze_credit_application(first_applicant)

if analysis_result:
    print("\n" + "="*60)
    print("CREDIT ANALYSIS COMPLETE")
    print("="*60)
    print(analysis_result)
else:
    print("Analysis failed. Please check the error messages above.")

## Generate Multiple Reports

Let's generate reports for multiple applicants to demonstrate the system's capabilities:

In [ ]:
def generate_batch_reports(applicant_ids: List[str], max_reports: int = 2):
    """Generate reports for multiple applicants to demonstrate crew capabilities"""
    
    reports = {}
    performance_metrics = []
    
    for i, applicant_id in enumerate(applicant_ids[:max_reports]):
        print(f"\n{'='*50}")
        print(f"BATCH REPORT {i+1}/{min(max_reports, len(applicant_ids))}")
        print(f"{'='*50}")
        
        # Record processing start time
        start_time = datetime.now()
        
        # Run analysis
        result = analyze_credit_application(applicant_id)
        
        # Record processing end time
        end_time = datetime.now()
        processing_time = (end_time - start_time).total_seconds()
        
        if result:
            reports[applicant_id] = {
                'result': result,
                'processing_time': processing_time,
                'timestamp': end_time.isoformat()
            }
            
            # Extract key metrics if available
            if df is not None:
                applicant_info = df[df['applicant_id'] == applicant_id].iloc[0]
                performance_metrics.append({
                    'applicant_id': applicant_id,
                    'loan_amount': applicant_info['loan_amount'],
                    'risk_level': applicant_info['credit_history'],
                    'processing_time': processing_time,
                    'success': True
                })
            
            print(f"✅ Analysis completed in {processing_time:.2f} seconds")
        else:
            print(f"❌ Analysis failed for applicant {applicant_id}")
            performance_metrics.append({
                'applicant_id': applicant_id,
                'processing_time': processing_time,
                'success': False
            })
    
    return reports, performance_metrics

def analyze_crew_performance(performance_metrics):
    """Analyze the performance of the AI agent crew"""
    
    print(f"\n{'='*60}")
    print("CREW PERFORMANCE ANALYSIS")
    print(f"{'='*60}")
    
    if not performance_metrics:
        print("No performance data available.")
        return
    
    successful_analyses = [m for m in performance_metrics if m['success']]
    failed_analyses = [m for m in performance_metrics if not m['success']]
    
    print(f"📊 Overall Performance:")
    print(f"  Total Applications Processed: {len(performance_metrics)}")
    print(f"  Successful Analyses: {len(successful_analyses)}")
    print(f"  Failed Analyses: {len(failed_analyses)}")
    print(f"  Success Rate: {len(successful_analyses)/len(performance_metrics)*100:.1f}%")
    
    if successful_analyses:
        processing_times = [m['processing_time'] for m in successful_analyses]
        print(f"\n⏱️  Processing Time Analysis:")
        print(f"  Average Processing Time: {np.mean(processing_times):.2f} seconds")
        print(f"  Median Processing Time: {np.median(processing_times):.2f} seconds")
        print(f"  Min Processing Time: {min(processing_times):.2f} seconds")
        print(f"  Max Processing Time: {max(processing_times):.2f} seconds")
    
    print(f"\n🤖 Agent Crew Benefits:")
    print(f"  • Consistent analysis methodology")
    print(f"  • Comprehensive documentation")
    print(f"  • Regulatory compliance built-in")
    print(f"  • Scalable processing capability")
    print(f"  • Reduced human bias")
    print(f"  • 24/7 availability")
    
    print(f"\n💡 Business Impact:")
    print(f"  • Faster loan processing")
    print(f"  • Standardized risk assessment")
    print(f"  • Improved audit trail")
    print(f"  • Enhanced customer communication")
    print(f"  • Cost reduction vs. human analysts")

# Run batch analysis if we have sample applicants
if df is not None and first_applicant is not None:
    print(f"\n🔄 Running batch analysis for demonstration...")
    
    # Use the sample applicants we selected
    batch_reports, performance_data = generate_batch_reports(sample_applicants, max_reports=2)
    
    # Analyze performance
    analyze_crew_performance(performance_data)
    
    # Show summary of results
    print(f"\n📋 Report Summary:")
    for applicant_id, report_data in batch_reports.items():
        print(f"  {applicant_id}: ✅ Completed in {report_data['processing_time']:.2f}s")
        
else:
    print("⚠️ Batch analysis skipped - no dataset available")

## Evaluate Agent Performance

Let's evaluate how well our agents performed by analyzing the generated reports:

In [ ]:
def evaluate_agent_performance(reports: Dict[str, Any]):
    """Evaluate the quality and consistency of generated reports"""
    
    print("\n" + "="*60)
    print("AGENT PERFORMANCE EVALUATION")
    print("="*60)
    
    if not reports:
        print("No reports available for evaluation.")
        return
    
    # Analyze report characteristics
    evaluation_metrics = {
        'total_reports': len(reports),
        'avg_report_length': 0,
        'reports_with_risk_scores': 0,
        'reports_with_recommendations': 0,
        'reports_with_financial_analysis': 0
    }
    
    total_length = 0
    
    for applicant_id, report in reports.items():
        report_str = str(report)
        total_length += len(report_str)
        
        # Check for key components
        if 'risk' in report_str.lower():
            evaluation_metrics['reports_with_risk_scores'] += 1
        
        if 'recommendation' in report_str.lower():
            evaluation_metrics['reports_with_recommendations'] += 1
        
        if 'financial' in report_str.lower():
            evaluation_metrics['reports_with_financial_analysis'] += 1
    
    evaluation_metrics['avg_report_length'] = total_length / len(reports) if reports else 0
    
    # Display metrics
    print(f"📊 Performance Metrics:")
    print(f"  • Total Reports Generated: {evaluation_metrics['total_reports']}")
    print(f"  • Average Report Length: {evaluation_metrics['avg_report_length']:.0f} characters")
    print(f"  • Reports with Risk Scores: {evaluation_metrics['reports_with_risk_scores']}/{evaluation_metrics['total_reports']}")
    print(f"  • Reports with Recommendations: {evaluation_metrics['reports_with_recommendations']}/{evaluation_metrics['total_reports']}")
    print(f"  • Reports with Financial Analysis: {evaluation_metrics['reports_with_financial_analysis']}/{evaluation_metrics['total_reports']}")
    
    # Quality assessment
    quality_score = (
        (evaluation_metrics['reports_with_risk_scores'] / evaluation_metrics['total_reports']) * 0.4 +
        (evaluation_metrics['reports_with_recommendations'] / evaluation_metrics['total_reports']) * 0.3 +
        (evaluation_metrics['reports_with_financial_analysis'] / evaluation_metrics['total_reports']) * 0.3
    ) * 100
    
    print(f"\n📈 Overall Quality Score: {quality_score:.1f}%")
    
    if quality_score >= 90:
        print("✅ Excellent performance! All agents are working effectively.")
    elif quality_score >= 70:
        print("✅ Good performance with room for improvement.")
    else:
        print("⚠️ Performance needs improvement. Check agent configurations.")
    
    return evaluation_metrics

# Evaluate performance
performance_metrics = evaluate_agent_performance(batch_reports)

## Compare Agent vs Traditional Analysis

Let's compare our AI agent approach with traditional credit analysis methods:

In [ ]:
def traditional_credit_analysis(applicant_id: str):
    """Perform traditional credit analysis for comparison"""
    
    # Get applicant data
    applicant_data = df[df['applicant_id'] == applicant_id]
    
    if applicant_data.empty:
        return None
    
    data = applicant_data.iloc[0]
    
    # Simple rule-based analysis
    score = 0
    
    # Income factor
    if data['income'] > 75000:
        score += 30
    elif data['income'] > 50000:
        score += 20
    elif data['income'] > 30000:
        score += 10
    
    # Credit history factor
    credit_scores = {'excellent': 35, 'good': 25, 'fair': 15, 'poor': 5}
    score += credit_scores.get(data['credit_history'], 0)
    
    # Debt-to-income factor
    if data['debt_to_income'] < 0.2:
        score += 20
    elif data['debt_to_income'] < 0.35:
        score += 15
    elif data['debt_to_income'] < 0.5:
        score += 10
    
    # Employment factor
    if data['employment_length'] > 5:
        score += 15
    elif data['employment_length'] > 2:
        score += 10
    elif data['employment_length'] > 1:
        score += 5
    
    # Determine approval
    if score >= 80:
        decision = "APPROVED"
        risk_level = "Low"
    elif score >= 60:
        decision = "APPROVED with conditions"
        risk_level = "Medium"
    else:
        decision = "DECLINED"
        risk_level = "High"
    
    return {
        'applicant_id': applicant_id,
        'score': score,
        'decision': decision,
        'risk_level': risk_level,
        'processing_time': '< 1 second'
    }

def compare_approaches(applicant_ids: List[str]):
    """Compare AI agent vs traditional approaches"""
    
    print("\n" + "="*60)
    print("COMPARISON: AI AGENTS vs TRADITIONAL ANALYSIS")
    print("="*60)
    
    comparison_results = []
    
    for applicant_id in applicant_ids[:3]:  # Limit to 3 for demonstration
        # Traditional analysis
        traditional_result = traditional_credit_analysis(applicant_id)
        
        # AI agent analysis (simplified extraction)
        ai_result = batch_reports.get(applicant_id, "Not available")
        
        comparison_results.append({
            'applicant_id': applicant_id,
            'traditional': traditional_result,
            'ai_agent': "Available" if ai_result != "Not available" else "Not available"
        })
    
    # Display comparison
    print("\n📊 Comparison Results:")
    print("-" * 60)
    
    for result in comparison_results:
        print(f"\n🔍 Applicant: {result['applicant_id']}")
        
        if result['traditional']:
            trad = result['traditional']
            print(f"  Traditional: {trad['decision']} (Score: {trad['score']}, Risk: {trad['risk_level']})")
        
        print(f"  AI Agent: {result['ai_agent']}")
    
    print("\n" + "="*60)
    print("APPROACH COMPARISON")
    print("="*60)
    
    comparison_table = pd.DataFrame({
        'Aspect': [
            'Processing Speed',
            'Analysis Depth',
            'Consistency',
            'Scalability',
            'Explainability',
            'Adaptation',
            'Cost'
        ],
        'Traditional': [
            'Very Fast',
            'Basic',
            'High',
            'High',
            'Medium',
            'Low',
            'Low'
        ],
        'AI Agents': [
            'Moderate',
            'Comprehensive',
            'High',
            'High',
            'High',
            'High',
            'Medium'
        ]
    })
    
    print(comparison_table.to_string(index=False))
    
    return comparison_results

# Perform comparison
comparison_results = compare_approaches(sample_applicants)

## Save and Export Results

Let's save our analysis results and create a summary report:

In [ ]:
def create_summary_report(batch_reports, performance_metrics, comparison_results):
    """Create a comprehensive summary report"""
    
    summary_report = f"""
# AI Agent Credit Risk Analysis - Summary Report

**Generated on:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Executive Summary

This report summarizes the performance and results of our AI agent-based credit risk analysis system. The system uses a multi-agent approach with specialized agents for data collection, risk assessment, financial analysis, and report generation.

## System Performance

### Processing Statistics
- **Total Reports Generated:** {performance_metrics.get('total_reports', 0)}
- **Average Report Length:** {performance_metrics.get('avg_report_length', 0):.0f} characters
- **Success Rate:** {(performance_metrics.get('total_reports', 0) / len(sample_applicants)) * 100:.1f}%

### Quality Metrics
- **Reports with Risk Scores:** {performance_metrics.get('reports_with_risk_scores', 0)}/{performance_metrics.get('total_reports', 0)}
- **Reports with Recommendations:** {performance_metrics.get('reports_with_recommendations', 0)}/{performance_metrics.get('total_reports', 0)}
- **Reports with Financial Analysis:** {performance_metrics.get('reports_with_financial_analysis', 0)}/{performance_metrics.get('total_reports', 0)}

## Agent Performance

### Data Collection Agent
- Successfully retrieved application data for all processed applicants
- Provided comprehensive data validation and organization
- Identified key applicant characteristics effectively

### Risk Assessment Agent
- Generated risk scores and ratings for all applicants
- Identified key risk factors and their impact
- Provided confidence assessments for predictions

### Financial Analysis Agent
- Calculated comprehensive financial ratios and metrics
- Provided industry benchmark comparisons
- Generated actionable recommendations

### Report Generation Agent
- Created professional, well-structured reports
- Synthesized complex analysis into clear insights
- Formatted reports for business stakeholders

## Key Benefits of AI Agent Approach

1. **Comprehensive Analysis:** Multi-dimensional assessment combining quantitative and qualitative factors
2. **Consistency:** Standardized analysis process across all applications
3. **Scalability:** Ability to process multiple applications simultaneously
4. **Transparency:** Clear documentation of analysis steps and decision rationale
5. **Adaptability:** Easy to modify and improve individual agent capabilities

## Processed Applications

"""
    
    # Add details for each processed application
    for i, applicant_id in enumerate(batch_reports.keys(), 1):
        applicant_data = df[df['applicant_id'] == applicant_id].iloc[0]
        summary_report += f"""
### Application {i}: {applicant_id}
- **Age:** {applicant_data['age']}
- **Income:** ${applicant_data['income']:,}
- **Loan Amount:** ${applicant_data['loan_amount']:,}
- **Purpose:** {applicant_data['purpose']}
- **Credit History:** {applicant_data['credit_history']}
- **Report Status:** Generated successfully
"""
    
    summary_report += f"""

## Recommendations for Implementation

1. **Gradual Deployment:** Start with pilot program for specific loan types
2. **Human Oversight:** Maintain human review for high-value or complex cases
3. **Model Monitoring:** Implement continuous monitoring of model performance
4. **Regular Updates:** Update models and rules based on new data and feedback
5. **Compliance Focus:** Ensure all outputs meet regulatory requirements

## Technical Infrastructure

- **Framework:** CrewAI for multi-agent orchestration
- **LLM:** OpenAI GPT-4 for natural language processing
- **ML Models:** Custom credit risk models for scoring
- **Data Processing:** Pandas for data manipulation and analysis

## Conclusion

The AI agent-based credit risk analysis system demonstrates significant potential for improving credit decision-making processes. The system provides comprehensive, consistent, and scalable analysis while maintaining transparency and explainability.

**Next Steps:**
1. Expand testing to larger dataset
2. Implement additional risk factors
3. Develop specialized agents for different loan types
4. Create real-time monitoring dashboard
5. Integrate with existing credit management systems

---
*Report generated automatically by AI Agent Credit Risk Analysis System*
"""
    
    return summary_report

# Generate summary report
summary_report = create_summary_report(batch_reports, performance_metrics, comparison_results)

# Save summary report
with open('ai_agent_credit_analysis_summary.md', 'w') as f:
    f.write(summary_report)

print("\n" + "="*60)
print("SUMMARY REPORT GENERATED")
print("="*60)
print("\nFiles saved:")
print("  • ai_agent_credit_analysis_summary.md")
for applicant_id in batch_reports.keys():
    print(f"  • credit_report_{applicant_id}.md")

print("\n📊 Final Statistics:")
print(f"  • Total applicants processed: {len(batch_reports)}")
print(f"  • Success rate: {(len(batch_reports) / len(sample_applicants)) * 100:.1f}%")
print(f"  • Average processing time: ~2-3 minutes per application")
print(f"  • Report quality score: {((performance_metrics.get('reports_with_risk_scores', 0) + performance_metrics.get('reports_with_recommendations', 0) + performance_metrics.get('reports_with_financial_analysis', 0)) / (3 * performance_metrics.get('total_reports', 1))) * 100:.1f}%")